# Analyze results from LLM Annotation of Applied Prognostic Evidence Line Comments

LLM metadata
- model_id: us.anthropic.claude-sonnet-4-6
- temperature: 0.0

In [ ]:
import math
from collections.abc import Mapping
from enum import StrEnum
from pathlib import Path
from typing import Any

import polars as pl
from pydantic import BaseModel, ConfigDict
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import BasePromptTemplate, build_empty_registry
from wags_llm.services import StructuredTaskRunner

In [2]:
# This file includes human annotations for analysis
SAMPLE_PATH = Path(
    "cleaned_public-sample_prognostic_annotated_ingested_vafs_2026-04-28.tsv"
)

In [3]:
PROMPT_NAME = "evidence_direction_annotation:prognostic"
PROMPT_VERSION = "v1"


class PrognosticDirectionPromptV1(BasePromptTemplate):
    """Version 1 prompt for interpreting prognostic evidence direction from comments."""

    version = PROMPT_VERSION
    name = PROMPT_NAME

    def build_system_prompt(self) -> str:
        """Build the system prompt for extracting prognostic evidence direction from a
        VAF evidence description and comment.

        :returns: System prompt text.
        """
        return (
            "Task:\n"
            "Given an evidence description and comments, determine the clinical impact direction for prognosis for ClinVar AMP/ASCO/CAP assertionTypeForClinicalImpact submission. Comments may include identifiers, analyst, and/or director (in order; may be missing).\n\n"
            "Rules:\n"
            "- Ignore identifiers\n"
            "- Use the last comment; if minimal, use the nearest preceding substantive comment(s). These are the governing comment(s)\n"
            "- Use description for prognostic applicability; the governing comment(s) may confirm or override it\n"
            "- If the governing comment(s) indicate non-applicability, exclusion, or insufficient support, return absent\n"
            "- unclear = prognosis mentioned but direction is ambiguous or uncertain\n"
            "- absent = prognosis does not apply or is not mentioned\n"
        )

    def build_user_prompt(
        self,
        payload: Mapping[str, Any],
    ) -> str:
        """Build the user prompt for a single comment.

        :param payload: Evidence direction and comments within VAF evidence line
        :returns: User prompt text
        """
        return (
            f"Evidence Description: {payload['description']}\n"
            f"Comments: {payload['comments']}\n"
        )

In [4]:
class PrognosticDirection(StrEnum):
    """Define directionality for prognostic evidence"""

    BETTER = "better outcome"
    POOR = "poor outcome"
    UNCLEAR = "unclear"
    ABSENT = "absent"


class PrognosticEvidenceDirectionResult(BaseModel):
    """Model for LLM and human curator Result for extracting prognostic direction from
    VAF sheet comments
    """

    model_config = ConfigDict(extra="forbid", use_enum_values=True)

    evidence_line_type_direction: PrognosticDirection

In [ ]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150
TEMPERATURE = 0.0


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM evidence direction annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(PrognosticDirectionPromptV1())
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )


task_runner = build_llm_task_runner(
    MODEL_ID, REGION_NAME, PROFILE_NAME, MAX_TOKENS, TEMPERATURE
)

In [7]:
response_model = PrognosticEvidenceDirectionResult
df = pl.read_csv(SAMPLE_PATH, separator="\t")

annotation_values = []
error_messages = []

In [ ]:
for _i, row in enumerate(
    tqdm(df.iter_rows(named=True), total=df.height, desc="Processing rows")
):
    comments = row["Comments"] or []
    description = row["Description"]

    try:
        task_result = task_runner.execute(
            prompt_name=PROMPT_NAME,
            prompt_version=PROMPT_VERSION,
            payload={
                "description": description,
                "comments": comments,
            },
            response_model=response_model,
        )

    except Exception as e:
        annotation_value = "Error"
        error_msg = str(e)

    else:
        annotation = response_model.model_validate(task_result)
        annotation_value = annotation.evidence_line_type_direction
        error_msg = None

    annotation_values.append(annotation_value)
    error_messages.append(error_msg)

Processing rows:   0%|          | 0/163 [00:00<?, ?it/s]

In [9]:
df = df.with_columns(
    [
        pl.Series("LLM Annotation", annotation_values),
        pl.Series("LLM error message", error_messages),
    ]
)

In [10]:
df

ID,Evidence Line Type,Description,Requirement Met,Comments,Curator Annotation,Curator Note,LLM Annotation,LLM error message
i64,str,str,bool,str,str,str,str,null
0,"""prognostic""","""Provides prognostic informatio…",true,"""['34796414', 'Associated with …","""better outcome""",null,"""better outcome""",null
1,"""prognostic""","""Prognostic significance based …",true,"""['31480372, 34952640', 'Yes, h…","""better outcome""",null,"""better outcome""",null
2,"""prognostic""","""Provides prognostic informatio…",true,"""['34796414', 'Associated with …","""unclear""",null,"""better outcome""",null
3,"""prognostic""","""Prognostic significance based …",true,"""['28069929', 'Some evidence th…","""poor outcome""",null,"""poor outcome""",null
4,"""prognostic""","""Prognostic significance based …",true,"""['32878261, 36950649, 38130310…","""absent""",null,"""absent""",null
…,…,…,…,…,…,…,…,…
156,"""prognostic""","""Provides prognostic informatio…",true,"""['16076867, 17440048, 16455956…","""better outcome""",null,"""better outcome""",null
157,"""prognostic""","""Provides prognostic informatio…",true,"""['16258095, 30765705, 20921458…","""better outcome""",null,"""better outcome""",null
158,"""prognostic""","""Provides prognostic informatio…",true,"""['34796414', 'Associated with …","""better outcome""",null,"""better outcome""",null


## Helper functions

In [11]:
def confusion_matrix_with_collapse(
    df: pl.DataFrame,
    gt_col: str,
    pred_col: str,
    collapse: bool = False,
) -> pl.DataFrame:
    """Compute a confusion matrix for:
    ['poor outcome', 'better outcome', 'unclear', 'absent'],
    with optional merging of 'unclear' and 'absent' into 'indeterminate'.

    :param df: Input DataFrame with evidence lines, associated comments, curator annotations, and LLM annotations
    :param gt_col: ground truth/human curator values column name (e.g., 'Curator Annotation')
    :param pred_col: predicted column name (e.g., 'LLM Annotation')
    :param collapse: whether to collapse 'unclear' and 'absent' into 'indeterminate'
    :return: confusion matrix as a DataFrame
    """
    # Normalize ground truth
    gt = (
        pl.col(gt_col)
        .fill_null("absent")
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
    )
    # Normalize predictions
    pred = (
        pl.col(pred_col)
        .fill_null("absent")
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
    )

    working_df = df.with_columns(
        [
            gt.alias("gt"),
            pred.alias("pred"),
        ]
    )

    if collapse:
        collapse_map = {
            "unclear": "indeterminate",
            "absent": "indeterminate",
        }

        working_df = working_df.with_columns(
            [
                pl.col("gt").replace(collapse_map),
                pl.col("pred").replace(collapse_map),
            ]
        )

        classes = [
            "poor outcome",
            "better outcome",
            "indeterminate",
        ]

    else:
        classes = [
            "poor outcome",
            "better outcome",
            "unclear",
            "absent",
        ]

    cm = (
        working_df.group_by(["gt", "pred"])
        .len()
        .pivot(
            values="len",
            index="gt",
            on="pred",
        )
        .fill_null(0)
    )

    # Ensure all rows exist
    missing_rows = [cls for cls in classes if cls not in cm["gt"].to_list()]

    if missing_rows:
        cm = cm.with_columns(pl.col("gt").cast(pl.Utf8))

        cm = pl.concat(
            [
                cm,
                pl.DataFrame(
                    {
                        "gt": pl.Series(missing_rows, dtype=pl.Utf8),
                        **{
                            c: pl.Series(
                                [0] * len(missing_rows),
                                dtype=cm.schema.get(c, pl.UInt32),
                            )
                            for c in classes
                        },
                    }
                ),
            ],
            how="diagonal",
        )

    # Ensure all columns exist
    for cls in classes:
        if cls not in cm.columns:
            cm = cm.with_columns(pl.lit(0).alias(cls))

    # Reorder
    cm = cm.select(["gt", *classes])

    # Sort rows
    cm = (
        cm.with_columns(
            pl.col("gt").replace({v: i for i, v in enumerate(classes)}).alias("_order")
        )
        .sort("_order")
        .drop("_order")
    )

    return cm.rename({"gt": "Ground Truth"})

In [12]:
def create_analysis_summary(cm: pl.DataFrame) -> pl.DataFrame:
    """Compute per-class recall-style summary from a confusion matrix.

    :param cm: Confusion matrix (square DataFrame)
    :return: Summary DataFrame per class
    """
    rows = []

    classes = cm.columns[1:]

    for cls in classes:
        row = cm.filter(pl.col("Ground Truth") == cls)

        if row.height == 0:
            numerator = 0
            denominator = 0
        else:
            numerator = row.select(pl.col(cls)).item()

            denominator = row.select(pl.sum_horizontal(classes)).item()

        rows.append(
            {
                "consensus_w_curator": cls,
                "numerator": numerator,
                "denominator": denominator,
                "percentage": (
                    (numerator or 0) / denominator * 100
                    if denominator is not None and denominator > 0
                    else 0.0
                ),
            }
        )

    return pl.DataFrame(rows)

In [13]:
def compute_overall_accuracy(cm: pl.DataFrame) -> float:
    """Compute overall accuracy from a confusion matrix.

    :param cm: Confusion matrix DataFrame (square matrix)
    :return: Accuracy as a float between 0 and 1
    """
    classes = cm.columns[1:]

    total = cm.select(pl.sum_horizontal(classes)).to_series().sum()

    correct = 0

    for cls in classes:
        row = cm.filter(pl.col("Ground Truth") == cls)

        if row.height > 0:
            correct += row.select(pl.col(cls)).item() or 0

    return correct / total if total > 0 else math.nan

In [14]:
def analyze_results(
    df: pl.DataFrame,
    collapse: bool = True,
) -> tuple[pl.DataFrame, float, pl.DataFrame]:
    """Evaluate LLM predictions against ground truth using a confusion matrix.

    Computes:
      - confusion matrix (optionally collapsed classes)
      - per-class performance summary (recall-style)
      - overall accuracy

    :param df: Input DataFrame with ground truth and predictions
    :param collapse: If True, merges 'unclear' and 'absent' into 'indeterminate'
    :return:
        - match_analysis_summary: per-class recall summary
        - accuracy: overall accuracy
        - cm: confusion matrix
    """
    analysis_df = df.clone()

    cm = confusion_matrix_with_collapse(
        analysis_df,
        "Curator Annotation",
        "LLM Annotation",
        collapse=collapse,
    )

    match_analysis_summary = create_analysis_summary(cm)

    accuracy = compute_overall_accuracy(cm)

    return match_analysis_summary, accuracy, cm

In [17]:
def process_results(
    df: pl.DataFrame,
    collapse: bool,
) -> tuple[dict, dict, list]:
    """Run analysis on a provided model output DataFrame.

    :param df: Input DataFrame containing model results
    :param collapse: Whether to collapse categories during analysis
    :return: Tuple of all results (summaries, cm, accuracies)
    """
    summary, accuracy, cm = analyze_results(
        df,
        collapse=collapse,
    )

    return (
        summary,
        cm,
        accuracy,
    )

## Analysis of Runs

### Main paper

In [18]:
summary, cm, accuracy = process_results(df, collapse=True)

In [19]:
cm

Ground Truth,poor outcome,better outcome,indeterminate
str,u32,u32,u32
"""poor outcome""",50,1,1
"""better outcome""",0,50,0
"""indeterminate""",24,8,29


In [20]:
summary

consensus_w_curator,numerator,denominator,percentage
str,i64,i64,f64
"""poor outcome""",50,52,96.153846
"""better outcome""",50,50,100.0
"""indeterminate""",29,61,47.540984


In [21]:
accuracy

0.7914110429447853

### Supplemental

In [22]:
sup_summary, sup_cm, sup_accuracy = process_results(df, collapse=False)

In [23]:
sup_cm

Ground Truth,poor outcome,better outcome,unclear,absent
str,u32,u32,u32,u32
"""poor outcome""",50,1,1,0
"""better outcome""",0,50,0,0
"""unclear""",6,6,5,1
"""absent""",18,2,16,7


In [24]:
sup_summary

consensus_w_curator,numerator,denominator,percentage
str,i64,i64,f64
"""poor outcome""",50,52,96.153846
"""better outcome""",50,50,100.0
"""unclear""",5,18,27.777778
"""absent""",7,43,16.27907


In [25]:
sup_accuracy

0.6871165644171779